[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MagedSaeed/instructions-tuning/blob/main/Notebooks/Experiments/Baselines/emotion_detection_tuning.ipynb)

# Get corekit from GitHub

coregit is a library of useful utilities that we can utilize for generation and tuning

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [3]:
from dotenv import load_dotenv
load_dotenv()

False

In [4]:
# !echo $GT_TOKEN # makesure the token is loaded

In [5]:
if 'jrcai_corekit' not in os.listdir('.'):
    !git clone https://${GT_TOKEN}@github.com/MagedSaeed/jrcai_corekit.git
else: # else, pull latest changes
    !cd jrcai_corekit && git pull && cd ..

Already up to date.


In [6]:
!pip install -r jrcai_corekit/requirements.txt

/bin/bash: /home/majed_alshaibani/Projects/instructions-tuning/venv/bin/pip: /home/majed_alshaibani/Projects/InstructionsTuning/venv/bin/python3: bad interpreter: No such file or directory


add jrcai_corekit to path

In [7]:
import sys
sys.path.append('jrcai_corekit/src')

check everything is working

In [8]:
from llm.text_generator import TextGenerator

/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/deepspeed.py:24: FutureWarning: transformers.deepspeed module is deprecated and will be removed in a future version. Please import deepspeed modules directly from transformers.integrations
  warnings.warn(


# Constants

In [9]:
TAWJEEH_DATASET_NAME = 'AraBench_dev'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/arabench_dev_experimental'
TASK_NAME='dialect_identification'
MODEL_PATH = "/hdd/shared_models/AceGPT-7B"

In [10]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [11]:
import requests

from tqdm.auto import tqdm

prompts = None

tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://tawjeeh.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts: raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14892,
  'tags': [],
  'name': 'mais-prompt1',
  'task': {'name': 'NLI'},
  'status': 'SUBMITTED',
  'template': 'Premise: {{ Premise }}\r\nHypothesis: {{ Hypothesis }}\r\n\r\nDoes the hypothesis about the premise entails? (Yes or No)\r\nAnswer:\r\n|||\r\n{{ answer_choices[label] }}',
  'created_by': 'mais',
  'dataset_name': 'arbml/ArabicTE',
  'dataset_subset': 'default',
  'answer_choices': ['No', 'Yes'],
  'text_direction': 'ltr'},
 {'id': 14891,
  'tags': [],
  'name': 'expert Arabic summarizer',
  'task': {'name': 'summarization'},
  'status': 'APPROVED',
  'template': 'You are an expert Arabic text summarizer. The following article:\r\n{{article}}\xa0\r\ncan be summarized as:\r\n|||\r\n{{summary}}',
  'created_by': 'majed.alshaibani',
  'dataset_name': 'arbml/AraSum',
  'dataset_subset': 'default',
  'answer_choices': [],
  'text_direction': 'ltr'},
 {'id': 14890,
  'tags': [],
  'name': 'Translation as completion',
  'task': {'name': 'machine translation'},
  'status': 

In [12]:
len(prompts)

358

In [13]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

235

## Finetuning

### Get the dataset prompts

In [14]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

6

In [15]:
SELECTED_PROMPTS_IDS = [
    14852,   
    14850,
    14789,
    14781,
    14561,
]

In [16]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the experimental dataset from HF

In [17]:
import datasets

In [18]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['arabic', 'english', 'label'],
        num_rows: 30000
    })
    test: Dataset({
        features: ['arabic', 'english', 'label'],
        num_rows: 10000
    })
})

### Merge the prompts

In [19]:
from jinja2 import Environment, StrictUndefined

In [20]:
import re
def preprocess_template(template):
    # remove punc at the end
    prefix,suffix = template.split('|||')
    prefix = prefix.replace('\xa0', '')
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    return f'{prefix.strip()}\n{suffix.strip()}' # output is always the last line!

In [21]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        template = preprocess_template(template)
        sample['answer_choices'] = prompt_template['answer_choices']
        env = Environment(undefined=StrictUndefined)
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

see how the template is applied on different examples

### Perform prompt-merge on one example prompt, for experimentation

In [22]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['train'][5]))

Consider this text:
اكا الباص جاي. احسن لك تستعجل.
If it is written as Modern-Standard-Arabic (MSA), then answer with MSA. If it is not an MSA text, then consider the following:
- If you think it is from Eastern Arabic countries, then answer with what you think is the closest, either Qatari or Lebanese.
- If you think it is from Western Arabic countries, then answer with what you think is the closest, either Tunisian, Morrocan, or Egyptian.
The answer is:
Qatari


In [23]:
step_size = len(hf_exp_dataset['train'])/len(dataset_prompts)
step_size

6000.0

In [24]:
rendered_train_prompts_dataset = list()
for i,sample in enumerate(tqdm(hf_exp_dataset['train'])):
    if i % step_size == 0:
        print(f'rending {dataset_prompts[int(i/step_size)]["template"]}','sample index:',i)
    rendered_train_prompts_dataset.append(
        apply_template(dataset_prompts[int(i/step_size)], sample)
    )
len(rendered_train_prompts_dataset)

  0%|          | 0/30000 [00:00<?, ?it/s]

rending Consider this text:
{{arabic}} 
If it is written as Modern-Standard-Arabic (MSA), then answer with MSA. If it is not an MSA text, then consider the following:
- If you think it is from Eastern Arabic countries, then answer with what you think is the closest, either Qatari or Lebanese.
- If you think it is from Western Arabic countries, then answer with what you think is the closest, either Tunisian, Morrocan, or Egyptian.
The answer is:
|||
{{answer_choices[label]}} sample index: 0
rending You're tasked with identifying wether a given Arabic phrase is MSA or dialectical, and if not MSA, what's the dialect of it.
Your answer must be one of the following choices: {% for choice in answer_choices %}{{ choice }}{% if not loop.last %}, {% endif %}{% endfor %}.
---
Sentence: {{arabic}}
Answer:
|||
{{answer_choices[label]}} sample index: 6000
rending Using your experience and knowledge of Arabic, can you infer the dialect used in the following text:
{{arabic}} 
You can only select from

30000

## Finetune the LLM

In [25]:
GLOBAL_SEED = 42

In [26]:
import random
random.seed(GLOBAL_SEED)

In [27]:
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from llm import train_llm, LLMLoader, Llama3Initializer,LoRAConfigRepository
from sklearn.model_selection import train_test_split

In [28]:
llm_loader = LLMLoader(
        MODEL_PATH,
        llm_initializer=Llama3Initializer(),
)
llm_loader

In [28]:
model, tokenizer, generation_config = llm_loader()

loading configuration file /hdd/shared_models/AceGPT-7B/config.json
Model config LlamaConfig {
  "_name_or_path": "/hdd/shared_models/AceGPT-7B",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_length": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 32000
}

loading weights file /hdd/shared_models/AceGPT-7B/pytorch_model.bin
Instantiating LlamaForCausalLM model under default dtype torch.bfl

In [ ]:
import re

train_samples,eval_samples = train_test_split(
    rendered_train_prompts_dataset,
    test_size=0.1,
    random_state=GLOBAL_SEED,
)

def generate_tuple(sample):
    sample_lines = sample.splitlines()
    prefix = '\n'.join(sample_lines[:-1])
    prefix = prefix.replace('\xa0', '')
    prefix = prefix.strip()
    suffix = sample_lines[-1].strip()
    suffix = f' {suffix}' # adding this space is important to split between input and output
    return prefix,suffix

train_samples = list(map(generate_tuple,train_samples))
eval_samples = list(map(generate_tuple,eval_samples))
len(train_samples), len(eval_samples), train_samples[:5], eval_samples[:5]

(27000,
 3000,
 [('Consider this text:\nترتفع الاسعار الى اس الى الى ارقام قياسيه\nIf it is written as Modern-Standard-Arabic (MSA), then answer with MSA. If it is not an MSA text, then consider the following:\n- If you think it is from Eastern Arabic countries, then answer with what you think is the closest, either Qatari or Lebanese.\n- If you think it is from Western Arabic countries, then answer with what you think is the closest, either Tunisian, Morrocan, or Egyptian.\nThe answer is:',
   ' Qatari'),
  ('Using your experience and knowledge of Arabic, can you infer the dialect used in the following text:\nعندي هيدا .\nYou can only select from the following dialects:\nTunisian,  MSA,  Morrocan,  Qatari,  Egyptian,  Lebanese.',
   ' Lebanese'),
  ("You're tasked with identifying wether a given Arabic phrase is MSA or dialectical, and if not MSA, what's the dialect of it.\nYour answer must be one of the following choices: Tunisian, MSA, Morrocan, Qatari, Egyptian, Lebanese.\n---\nSen

In [38]:
# save prompt samples
import json

# create a folder to save the prompts
import os
if not os.path.exists(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning'):
    os.makedirs(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning')


with open(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning/train.json','w') as f:
    json.dump(
        train_samples,
        f,
        indent=4,
        ensure_ascii=False,
    )

with open(f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/AceGPT/samples_used_for_tuning/val.json','w') as f:
    json.dump(
        eval_samples,
        f,
        indent=4,
        ensure_ascii=False,
    )

In [31]:
train_llm(
    model=model,
    tokenizer=tokenizer,
    train_samples=train_samples,
    eval_samples=eval_samples,
    peft_config=LoRAConfigRepository.llama_3(),
    learning_rate=2.5e-4,
    epochs_count=10,
    train_batch_size=16,
    eval_batch_size=16,
    output_dir=f'Notebooks/Experiments/{TASK_NAME}/{TAWJEEH_DATASET_NAME}/tuned_models/{MODEL_NAME}'
)

peft config LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path=None, revision=None, task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, inference_mode=False, r=16, target_modules={'q_proj', 'v_proj'}, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))


/home/majed_alshaibani/Projects/instructions-tuning/venv/lib/python3.10/site-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
You have loaded a model on multiple GPUs. `is_model_parallel` attribute will be force-set to `True` to avoid any unexpected behavior such as device placement mismatching.
Using auto half precision backend

***** Running Evaluation *****
  Num examples = 3000
  Batch size = 16


loading configuration file /hdd/shared_models/AceGPT-7B/config.json
Model config LlamaConfig {
  "_name_or_path": "meta-llama/Llama-2-7b-hf",
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "max_length": 4096,
  "max_position_embeddings": 2048,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pad_token_id": 0,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 10000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.45.2",
  "use_cache": true,
  "vocab_size": 32000
}



{'eval_loss': 3.997736930847168, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 51.7981, 'eval_samples_per_second': 57.917, 'eval_steps_per_second': 3.629}


***** Running training *****
  Num examples = 27,000
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 16,880
  Number of trainable parameters = 8,388,608


...

{'eval_loss': 0.060160208493471146, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 45.2516, 'eval_samples_per_second': 66.296, 'eval_steps_per_second': 4.155, 'epoch': 2.5177725118483414}



***** Running Evaluation *****
  Num examples = 3000
  Batch size = 16


{'eval_loss': 0.054052919149398804, 'eval_model_preparation_time': 0.0003, 'eval_runtime': 45.233, 'eval_samples_per_second': 66.323, 'eval_steps_per_second': 4.156, 'epoch': 2.665876777251185}




Training completed. Do not forget to share your model on huggingface.co/models =)




0.046335674822330475

In [ ]:
exit()

: 